# Assignment 2 — Version B

 **This is technically one assignment, but it covers a lot of ground — take it step by step, understand what each part does, and enjoy the process!**


# Step 1: Loading Your Data

Loading data is where every RAG pipeline begins. Before we can search or answer anything, we need to pull raw information from files — PDFs, databases, websites, etc. — and turn them into structured objects our system can work with. Each object holds the actual text content along with metadata (extra info like the file name and page number).


For now we are only working with PDFs. In the final project, we will handle many more formats: CSV, XLSX, DOCX, and TXT files too.
Feel free to try loading one extra format for practice — that would be great to see!

In [ ]:
import os
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader

In [ ]:
def load_all_pdfs(directory_path):

    collected_docs = []
    pdf_files = Path(directory_path).glob("**/*.pdf")
    for pdf_file in pdf_files:                  
        loader = PyPDFLoader(str(pdf_file))
        documents = loader.load()  
        for doc in documents:
            doc.metadata["filename"] = pdf_file.name
            doc.metadata["source_path"] = str(pdf_file) 
            if "page" in doc.metadata:
                doc.metadata["page_number"] = doc.metadata["page"]
            collected_docs.append(doc)
    return collected_docs

In [ ]:
loaded_pdf_docs = load_all_pdfs("pdf")
loaded_pdf_docs

# Step 2: Splitting Documents into Chunks

Once the documents are loaded, we need to break them into smaller pieces. This is called chunking. LLMs have a maximum input size — they can only look at so much text at once. Smaller chunks also help retrieve more precise information. We use **RecursiveCharacterTextSplitter** which tries to split at natural boundaries (paragraphs, then lines, then words) and maintains a small overlap between chunks so context isn't lost at the edges.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
def chunk_documents(documents, chunk_size=1000, chunk_overlap=200):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=['\n\n', '\n', ' ', '']
    )
    chunks = splitter.split_documents(documents)
    return chunks

In [ ]:
doc_chunks = chunk_documents(loaded_pdf_docs)
doc_chunks

# Step 3: Creating Embeddings

Embeddings turn text into numbers — specifically, lists of decimal numbers called vectors. These vectors encode the meaning of the text mathematically. Text with similar meanings will have vectors that are numerically close to each other. We use a pre-trained model called `all-MiniLM-L6-v2` from the `sentence-transformers` library, which runs locally without needing an API.

---

**`from sentence_transformers import SentenceTransformer`**

This line imports the embedding model class. The library:
- Downloads pretrained transformer models from HuggingFace
- Converts any text into fixed-size embedding vectors

It is built on top of HuggingFace Transformers and PyTorch.

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid 
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
class EmbeddingManager:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        print(f"Initializing model: {self.model_name}")
        self.model = SentenceTransformer(self.model_name)
        print("Model ready")
        embed_dim = self.model.get_sentence_embedding_dimension()

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        if self.model is None:
            raise ValueError("Embedding model has not been loaded yet")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        return embeddings

# Step 4: Storing Embeddings in a Vector Database

A Vector Database is purpose-built for storing and searching high-dimensional vectors. After we generate embeddings for all our chunks, we store them here so we can search them later. ChromaDB is our choice — it runs locally, persists data to disk, and lets us do fast nearest-neighbor searches without a cloud service.

In [ ]:
class VectorStore:
    """Handles storing and retrieving document embeddings using ChromaDB"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        os.makedirs(self.persist_directory, exist_ok=True)
        self.client = chromadb.PersistentClient(path=self.persist_directory)
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description": "PDF document embeddings"}
        )

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        if len(documents) != len(embeddings):
            raise ValueError("Document count and embedding count must be equal")
        ids = []
        metadatas = []
        docs = []
        embedding_list = []
        for i, doc in enumerate(documents):
            ids.append(uuid.uuid4().hex[:8])
            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)
            docs.append(doc.page_content)
            embedding_list.append(embeddings[i].tolist())
        self.collection.add(ids=ids, embeddings=embedding_list, metadatas=metadatas, documents=docs)

## Convert text chunks to embeddings and store them

In [ ]:
chunk_texts = [doc.page_content for doc in doc_chunks]
chunk_texts

In [ ]:
embedding_manager = EmbeddingManager()
embeddings = embedding_manager.generate_embeddings(chunk_texts)
vectorstore = VectorStore()
vectorstore.add_documents(doc_chunks, embeddings)

# Step 5: Querying the System

When a user types a question, we need to convert that question into an embedding too — using the exact same model — so it lives in the same vector space as our stored document chunks. That query embedding is then handed off to the vector database for search.

---

# Step 6: Ranking Results by Similarity

The vector database computes a similarity score (commonly cosine similarity) between the query vector and every stored document vector. It returns the top_k most similar chunks, ranked by score. We can set a minimum threshold to discard low-relevance results and keep only useful context.

In [ ]:
class RAGRetriever:
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        results = self.vector_store.collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=top_k
        )
        retrieved_docs = []
        num_results = len(results["ids"][0])
        for i in range(num_results):
            distance = results["distances"][0][i]
            sim_score = 1 - distance
            if sim_score >= score_threshold:
                retrieved_docs.append({
                    "id": results["ids"][0][i],
                    "content": results["documents"][0][i],
                    "metadata": results["metadatas"][0][i],
                    "similarity_score": sim_score,
                    "distance": distance,
                    "rank": i + 1
                })
        return retrieved_docs

In [ ]:
rag_retriever = RAGRetriever(vectorstore, embedding_manager)

In [ ]:
rag_retriever.retrieve("put something here similar to your data to retrieve")

# Step 7: Building the Prompt and Calling the LLM

This is where everything comes together. We take the retrieved document chunks, combine them into a context block, and write a prompt that instructs the LLM to answer the user's question using only that context. The LLM then generates a factually grounded answer rather than relying on its general training knowledge.

In [ ]:
def rag_simple(query, retriever, llm, top_k=3):
    retrieved_docs = retriever.retrieve(query=query, top_k=top_k)
    context = "\n\n".join(doc["content"] for doc in retrieved_docs)
    prompt = f"""Use the context below to answer the question.
Context: {context}
Question: {query}
Answer:
"""
    response = llm.invoke(prompt)
    return response.content

In [ ]:
answer = rag_simple("three reasons for forgetting", rag_retriever, llm)  # llm needs to be imported first
print(answer)

# Step 8: Advanced RAG Pipeline (Optional)

The advanced version adds several production-quality features: streaming output for real-time display, citation tracking so users know where each answer came from, a conversation history so the system remembers past questions, and optional summarization to condense long answers.

In [ ]:
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    retrieved_docs = retriever.retrieve(query=query, top_k=top_k, score_threshold=min_score)

    context = "\n\n".join(doc["content"] for doc in retrieved_docs)
    sources = []
    for doc in retrieved_docs:
        sources.append({
            "source_file": doc["metadata"].get("filename", "Unknown"),
            "page": doc["metadata"].get("page_number", "Unknown"),
            "sim_score": doc["similarity_score"],
            "preview": doc["content"][:150]
        })
    confidence = 0
    if retrieved_docs:
        confidence = max(doc["similarity_score"] for doc in retrieved_docs)

    prompt = f"""
Use the following context to answer the question.
Context: {context}
Question: {query}
Answer:
"""
    response = llm.invoke(prompt)
    result = {"answer": response.content, "sources": sources, "confidence": confidence}

    if return_context:
        result["context"] = context

    return result

In [ ]:
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        retrieved_docs = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        context = "\n\n".join(doc["content"] for doc in retrieved_docs)
        sources = []
        for doc in retrieved_docs:
            sources.append({
                "source_file": doc["metadata"].get("filename", "Unknown"),
                "page": doc["metadata"].get("page_number", "Unknown"),
                "sim_score": doc["similarity_score"]
            })

        prompt = f"""
Use the following context to answer the question.
Context: {context}
Question: {question}
Answer:
"""
        if stream:
            for i in range(0, len(prompt), 200):
                print(prompt[i:i+200])
                time.sleep(0.05)

        response = self.llm.invoke(prompt)
        answer = response.content
        citations = []
        for source in sources:
            citations.append(f"{source['source_file']} (Page {source['page']})")

        answer_with_citations = answer + "\n\nSources:\n" + "\n".join(citations)

        summary = None
        if summarize:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n\n{answer}"
            summary = self.llm.invoke(summary_prompt).content

        record = {"question": question, "answer": answer_with_citations, "sources": sources, "summary": summary}
        self.history.append(record)
        return {
            "question": question,
            "answer": answer_with_citations,
            "sources": sources,
            "summary": summary,
            "history": self.history
        }